# Faruq-v3 AF2-FT30 vs AF2-CAL3 — Kaggle screening

This notebook separates continuation-training gain from a three-parameter RGB residual calibration. Attach private dataset `faruq-v3-experiment-core-v1`, select **GPU T4 x2**, enable Internet, and run cells in order. It validates the opaque archive before any GPU training and never exposes test.

Do not stop the session after training. Run the decision and ZIP cells, then download `faruq-v3-af2-channel-calibration-output.zip` or save a Kaggle version.


In [ ]:
import importlib,importlib.metadata,json,os,shutil,subprocess,sys,time,torch
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
def unique(name):
    matches=sorted(path for path in INPUT.rglob(name) if path.is_file())
    if len(matches)!=1: raise FileNotFoundError(f'Harus ada tepat satu {name}; ditemukan {matches}')
    return matches[0]
af2_named=sorted(path for path in INPUT.rglob('AF2_seed42_best.pt') if path.is_file())
AF2=af2_named[0] if len(af2_named)==1 else unique('best.pt')
REPO=WORK/'coffee-bean-detection'; BRANCH='agent/af2-adaptive-residual-gate'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
TORCH_VERSION_BEFORE=importlib.metadata.version('torch')
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
TORCH_VERSION_AFTER=importlib.metadata.version('torch')
if TORCH_VERSION_AFTER!=TORCH_VERSION_BEFORE: raise RuntimeError(f'Instalasi mengubah Torch {TORCH_VERSION_BEFORE} -> {TORCH_VERSION_AFTER}; restart session.')
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import ultralytics
if ultralytics.__version__!='8.4.96': raise RuntimeError(f'Versi Ultralytics salah: {ultralytics.__version__}')
from coffee_detector.experiments.prepare_faruq_v3_kaggle import prepare_faruq_v3_kaggle_input
DATA,INPUT_CONTRACT=prepare_faruq_v3_kaggle_input(INPUT,WORK)
from ultralytics.data.utils import check_det_dataset
resolved=check_det_dataset(str(DATA/'data.yaml'))
if Path(resolved['train']).resolve()!=(DATA/'train/images').resolve(): raise RuntimeError(f'Ultralytics train path salah: {resolved["train"]}')
if Path(resolved['val']).resolve()!=(DATA/'val/images').resolve(): raise RuntimeError(f'Ultralytics val path salah: {resolved["val"]}')
assert torch.cuda.is_available(),'Aktifkan GPU T4 x2.'
GPU_NAME=torch.cuda.get_device_name(0); GPU_CAPABILITY=torch.cuda.get_device_capability(0)
if GPU_CAPABILITY[0]<7: raise RuntimeError(f'GPU {GPU_NAME} tidak kompatibel; pilih T4 x2.')
_cuda_probe=torch.ones(1,device='cuda:0').sum().item()
OUTPUT=WORK/'faruq-v3-af2-channel-calibration-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
(OUTPUT/'input_contract.json').write_text(json.dumps(INPUT_CONTRACT,indent=2)+'\n',encoding='utf-8')
EVIDENCE=REPO/'docs/evidence/FARUQ_V3_AF2R_SCREENING_2026-08-17.json'
print('GPU:',GPU_NAME); print('DATA:',DATA); print('AF2:',AF2); print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.af2cal.audit import run_af2cal_static_audit
STATIC=OUTPUT/'static_audit.json'
audit=run_af2cal_static_audit(AF2,STATIC,device='cpu')
print('PARAMETERS:',{'source':audit['source_parameters'],'candidate':audit['candidate_parameters'],'added':audit['added_parameters']})
print('GATES:',audit['gates']); print('DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal; jangan training.'


In [ ]:
def run_arm(arm):
    result_path=OUTPUT/'val_reports'/f'{arm}_seed42_result.json'
    if result_path.is_file():
        print('REUSE SELESAI:',arm); return json.loads(result_path.read_text())
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2cal_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
    log=OUTPUT/f'{arm}_seed42_run.log'; log.parent.mkdir(parents=True,exist_ok=True)
    print('START:',arm,'| log=',log,flush=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT)
        while process.poll() is None and not result_path.is_file():
            try: process.wait(timeout=300)
            except subprocess.TimeoutExpired:
                csv=OUTPUT/arm/f'{arm}_seed42/results.csv'; epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
                print(f'{arm}: {epochs}/30 epoch tercatat',flush=True)
        if result_path.is_file() and process.poll() is None:
            try: process.wait(timeout=60)
            except subprocess.TimeoutExpired:
                process.terminate(); process.wait(timeout=30)
    if not result_path.is_file():
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-150:])
        print(tail,flush=True); raise RuntimeError(f'{arm} gagal: {process.returncode}\n--- LOG TERAKHIR ---\n{tail}')
    print('SELESAI:',arm,flush=True); return json.loads(result_path.read_text())
results={arm:run_arm(arm) for arm in ('AF2FT30','AF2CAL3')}
print({arm:{k:v for k,v in result['metrics'].items() if k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')} for arm,result in results.items()})


In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2cal_decision import run_faruq_v3_af2cal_decision
decision=run_faruq_v3_af2cal_decision(OUTPUT,EVIDENCE,seed=42)
import pandas as pd
from IPython.display import display
rows=[{'model':model,**metrics} for model,metrics in decision['values'].items()]
display(pd.DataFrame(rows).style.format({c:'{:.2%}' for c in ['macro_map50_95','bottom3_class_map50_95','worst_class_map50_95']}))
print('CAL3 vs FT30:',decision['af2cal3_minus_af2ft30']); print('CAL3 vs AF2R0:',decision['af2cal3_minus_af2r0']); print('CRITERIA:',decision['criteria']); print('DECISION:',decision['decision']); print('ATTRIBUTION:',decision['attribution']); print('NEXT:',decision['next']); print('TEST:',decision['test_opened'])


In [ ]:
archive_path=shutil.make_archive(str(WORK/'faruq-v3-af2-channel-calibration-output'),'zip',root_dir=OUTPUT)
print('DOWNLOAD SEBELUM STOP SESSION:',archive_path)
print('Checkpoint, log, reports, static audit, dan decision ada di ZIP ini.')
